# 44 — Vĩ mô và thị trường

**Sản phẩm 4**, và notebook cuối cùng. Dashboard vĩ mô đặt cạnh VNINDEX: lạm
phát, tăng trưởng, tín dụng, tỷ giá, lãi suất, nghiệp vụ thị trường mở, và cán
cân thương mại.

`client.macro` là namespace **khác mọi namespace còn lại** ở đúng một chỗ, và
chỗ đó quyết định toàn bộ cách viết code:

> ⚠️ **`unit` ở đây là một CỘT, không phải thuộc tính của cả bảng.** Hỏi hai
> chỉ tiêu bất kỳ là có thể nhận `%` nằm cạnh `USD/thùng` trong cùng cột
> `value`. Đọc `unit` theo **từng dòng**.

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, duong, hom_nay, lui_ngay, thanh_doi_mau, ty_dong
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

## 1 · Tra mã trước, hỏi số sau

**Mã chỉ tiêu không đoán được từ tên.** `client.macro.series("cpi")` trả về
lỗi; mã thật là `chi_so_gia_tieu_dung_so_voi_cung_ky_nam_truoc_m_monthly`.
Luôn bắt đầu bằng `indicators()`.

In [2]:
danh_muc = client.macro.indicators()

print(f"{len(danh_muc):,} chuỗi chỉ tiêu · {danh_muc['topic'].nunique()} chủ đề")
print(f"Tần suất: {sorted(danh_muc['frequency'].unique())}\n")
danh_muc["topic"].value_counts().rename("số chuỗi").to_frame()

3,348 chuỗi chỉ tiêu · 24 chủ đề
Tần suất: ['daily', 'monthly', 'quarterly', 'yearly']



,số chuỗi
topic,
fdi,1765
nhap_khau,244
xuat_khau,226
chi_so_tieu_thu_nganh_cong_nghiep_che_bien_che_tao_iic,182
chi_so_san_xuat_cong_nghiep_iip,162
can_can_thanh_toan,140
chi_so_ton_kho_nganh_cong_nghiep_che_bien_che_tao_iii,126
thu_chi_ngan_sach_nha_nuoc,116
tong_san_pham_trong_nuoc_gdp,84


⚠️ Tham số `topic=` chỉ nhận đúng các giá trị trong cột `topic` ở trên. Gõ
`topic="cpi"` **không ném lỗi** — nó trả về một **bảng rỗng đúng schema**, và
đó là thứ dễ bị bỏ qua nhất.

In [3]:
rong = client.macro.indicators(topic="cpi")
print(f"topic='cpi'        → {len(rong)} dòng · cột vẫn đủ: {list(rong.columns)[:4]}…")

dung = client.macro.indicators(topic="chi_so_gia_tieu_dung_cpi")
print(f"topic='chi_so_gia_tieu_dung_cpi' → {len(dung)} dòng")

topic='cpi'        → 0 dòng · cột vẫn đủ: ['code', 'name', 'topic', 'frequency']…


topic='chi_so_gia_tieu_dung_cpi' → 80 dòng


### Tìm mã bằng cách lọc trên danh mục

Đây là cách làm bền: mô tả cái mình cần, để code tìm mã, rồi **khẳng định tìm
thấy**. Nếu nguồn đổi tên chuỗi, notebook dừng ngay với thông báo rõ ràng thay
vì chạy tiếp trên một bảng rỗng.

In [4]:
def tim_ma(mo_ta: str, *, topic: str | None = None, freq: str | None = None) -> str:
    """Tìm đúng MỘT mã chỉ tiêu khớp mô tả; ném lỗi nếu không tìm thấy hoặc mơ hồ."""
    d = danh_muc
    if topic:
        d = d[d["topic"] == topic]
    if freq:
        d = d[d["frequency"] == freq]
    d = d[d["name"].str.contains(mo_ta, case=False, na=False, regex=False)]
    if len(d) == 0:
        raise LookupError(f"Không tìm thấy chỉ tiêu nào khớp {mo_ta!r}")
    if len(d) > 1:
        raise LookupError(f"{len(d)} chỉ tiêu khớp {mo_ta!r}: {d['code'].tolist()[:5]}")
    return str(d["code"].iloc[0])


CHI_TIEU = {
    "CPI so cùng kỳ": tim_ma("Chỉ số giá tiêu dùng (So với cùng kỳ năm trước)"),
    "GDP tăng trưởng": tim_ma("Tổng GDP tăng trưởng YoY"),
    "Tín dụng so cùng kỳ": tim_ma("Tổng dư nợ tín dụng - % so với cùng kỳ"),
    "PMI": tim_ma("Chỉ số nhà quản trị mua hàng PMI"),
    "Tỷ giá trung tâm": tim_ma("Tỷ giá trung tâm"),
    "Lãi suất qua đêm": tim_ma("Lãi suất bình quân liên ngân hàng qua đêm"),
    "Vàng giao ngay": tim_ma("Giá vàng giao ngay"),
    "Dầu Brent": tim_ma("Dầu thô - Brent"),
}

for nhan, ma in CHI_TIEU.items():
    r = danh_muc[danh_muc["code"] == ma].iloc[0]
    print(f"{nhan:<22} {r['frequency']:<10} [{r['unit']:>9}]  {r['point_count']:>5,} điểm  → {r['last_date']:%d/%m/%Y}")

CPI so cùng kỳ         monthly    [        %]    118 điểm  → 31/07/2026
GDP tăng trưởng        quarterly  [        %]     38 điểm  → 30/06/2026
Tín dụng so cùng kỳ    monthly    [        %]    115 điểm  → 31/07/2026
PMI                    monthly    [      Lần]    116 điểm  → 31/07/2026
Tỷ giá trung tâm       daily      [      VND]  2,397 điểm  → 10/08/2026
Lãi suất qua đêm       daily      [        %]  2,352 điểm  → 07/08/2026
Vàng giao ngay         daily      [USD/Ounce]  2,452 điểm  → 10/08/2026
Dầu Brent              daily      [USD/thùng]  2,440 điểm  → 10/08/2026


## 2 · ⚠️ `unit` là một cột — bằng chứng

Hỏi cả tám chỉ tiêu trong một lời gọi và nhìn cột `unit`:

In [5]:
tat_ca = client.macro.series(list(CHI_TIEU.values()), start=lui_ngay(HOM_NAY, nam=3))

print(f"{len(tat_ca):,} dòng · {tat_ca['code'].nunique()} chuỗi\n")
print(tat_ca.groupby("code", observed=True).agg(
    ten=("name", "first"), don_vi=("unit", "first"), tan_suat=("frequency", "first"), so_diem=("value", "count")
).to_string())

3,164 dòng · 8 chuỗi

                                                                                                         ten     don_vi   tan_suat  so_diem
code                                                                                                                                       
chi_so_gia_tieu_dung_so_voi_cung_ky_nam_truoc_m_monthly  Chỉ số giá tiêu dùng (So với cùng kỳ năm trước) (M)          %    monthly       38
chi_so_nha_quan_tri_mua_hang_pmi_m_monthly                              Chỉ số nhà quản trị mua hàng PMI (M)        Lần    monthly       37
dau_tho_brent_daily                                                                          Dầu thô - Brent  USD/thùng      daily      773
gia_vang_giao_ngay_daily                                                                  Giá vàng giao ngay  USD/Ounce      daily      778
lai_suat_binh_quan_lien_ngan_hang_qua_dem_daily                    Lãi suất bình quân liên ngân hàng qua đêm          %      daily      74

In [6]:
print(f"Các đơn vị trong CÙNG một cột `value`: {sorted(tat_ca['unit'].unique())}")
print(f"\nTrung bình cột `value` của cả bảng: {tat_ca['value'].mean():,.1f}")
print("→ Con số đó trộn phần trăm với VND với USD/thùng. Nó không đo cái gì cả.")
print("\nCòn `df.attrs['finlens']['units']` thì nói gì?")
print(f"  {tat_ca.attrs['finlens']['units']}")
print("  → `value` không có đơn vị ở cấp bảng, đúng như thiết kế: nó nằm ở cột `unit`.")

Các đơn vị trong CÙNG một cột `value`: ['%', 'Lần', 'USD/Ounce', 'USD/thùng', 'VND']

Trung bình cột `value` của cả bảng: 6,610.8
→ Con số đó trộn phần trăm với VND với USD/thùng. Nó không đo cái gì cả.

Còn `df.attrs['finlens']['units']` thì nói gì?
  {'code': None, 'name': None, 'frequency': None, 'period': None, 'date': None, 'value': None, 'unit': None, 'is_cumulative': None}
  → `value` không có đơn vị ở cấp bảng, đúng như thiết kế: nó nằm ở cột `unit`.


**Quy tắc:** mọi phép tính trên `macro.series()` phải bắt đầu bằng một
`groupby("code")` hoặc một bộ lọc `code ==`. Không có ngoại lệ.

### Và cột `date` là cuối kỳ quan sát, không phải một phiên giao dịch

In [7]:
cpi = tat_ca[tat_ca["code"] == CHI_TIEU["CPI so cùng kỳ"]]
print("Ba dòng CPI gần nhất — chú ý cặp `period` và `date`:")
print(cpi.tail(3)[["period", "date", "value", "unit"]].to_string(index=False))
print("\n→ `period` nói kỳ nào ('7-2026'), `date` là ngày cuối kỳ đó (31/07/2026).")
print("  Đừng dùng `date` như một phiên giao dịch để merge với giá cổ phiếu.")

Ba dòng CPI gần nhất — chú ý cặp `period` và `date`:
period       date  value unit
5-2026 2026-05-29   5.60    %
6-2026 2026-06-30   4.69    %
7-2026 2026-07-31   4.45    %

→ `period` nói kỳ nào ('7-2026'), `date` là ngày cuối kỳ đó (31/07/2026).
  Đừng dùng `date` như một phiên giao dịch để merge với giá cổ phiếu.


## 3 · Lạm phát, tăng trưởng, tín dụng

In [8]:
def lay(nhan: str) -> pd.DataFrame:
    """Một chuỗi, đã lọc theo code — không bao giờ trộn đơn vị."""
    return tat_ca[tat_ca["code"] == CHI_TIEU[nhan]].sort_values("date")


fig = make_subplots(
    rows=3, cols=1, shared_xaxes=False, vertical_spacing=0.09,
    subplot_titles=("CPI so với cùng kỳ năm trước (%)", "Tăng trưởng GDP theo quý (%)",
                    "Tăng trưởng tín dụng so với cùng kỳ (%)"),
)
for hang, (nhan, mau) in enumerate(
    [("CPI so cùng kỳ", CHUOI[1]), ("GDP tăng trưởng", CHUOI[0]), ("Tín dụng so cùng kỳ", CHUOI[2])], start=1
):
    d = lay(nhan)
    fig.add_trace(
        go.Scatter(x=d["date"], y=d["value"], name=nhan, line=dict(width=2, color=mau),
                   hovertemplate="%{x|%m/%Y}: %{y:.2f}%<extra></extra>"),
        row=hang, col=1,
    )
fig.update_layout(
    title_text="Ba chỉ tiêu vĩ mô cốt lõi — 3 năm<br>"
    "<sub style='color:#52514e'>Ba khung riêng vì ba biên độ khác nhau; chồng lên một trục sẽ ép hai cái nằm bẹp</sub>",
    height=760, showlegend=False,
)
for i in range(1, 4):
    fig.update_yaxes(title_text="%", row=i, col=1)
fig

In [9]:
tom_tat = []
for nhan in ["CPI so cùng kỳ", "GDP tăng trưởng", "Tín dụng so cùng kỳ", "PMI"]:
    d = lay(nhan)
    if d.empty:
        continue
    tom_tat.append(
        {
            "chỉ tiêu": nhan,
            "kỳ gần nhất": d["period"].iloc[-1],
            "giá trị": round(float(d["value"].iloc[-1]), 2),
            "đơn vị": d["unit"].iloc[-1],
            "kỳ trước": round(float(d["value"].iloc[-2]), 2) if len(d) > 1 else np.nan,
            "12 kỳ trước": round(float(d["value"].iloc[-13]), 2) if len(d) > 13 else np.nan,
        }
    )
pd.DataFrame(tom_tat)

,chỉ tiêu,kỳ gần nhất,giá trị,đơn vị,kỳ trước,12 kỳ trước
0,CPI so cùng kỳ,7-2026,4.45,%,4.69,3.38
1,GDP tăng trưởng,Q2-2026,8.18,%,7.83,NaN
2,Tín dụng so cùng kỳ,7-2026,17.05,%,17.41,19.77
3,PMI,7-2026,52.90,Lần,51.80,52.40


## 4 · Tỷ giá và lãi suất — hai giá của tiền

In [10]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.5, 0.5])

tg = lay("Tỷ giá trung tâm")
ls = lay("Lãi suất qua đêm")

fig.add_trace(
    go.Scatter(x=tg["date"], y=tg["value"], name="Tỷ giá trung tâm",
               line=dict(width=2, color=CHUOI[0]), hovertemplate="%{y:,.0f} VND<extra></extra>"),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=ls["date"], y=ls["value"], name="Lãi suất qua đêm liên NH",
               line=dict(width=2, color=CHUOI[1]), hovertemplate="%{y:.2f}%<extra></extra>"),
    row=2, col=1,
)
fig.update_yaxes(title_text="VND / USD", row=1, col=1)
fig.update_yaxes(title_text="%/năm", row=2, col=1)
fig.update_layout(
    title_text="Tỷ giá trung tâm và lãi suất qua đêm<br>"
    "<sub style='color:#52514e'>Hai đại lượng, hai đơn vị, hai khung — trục thời gian chung</sub>",
    height=640, hovermode="x unified",
)
fig

## 5 · Nghiệp vụ thị trường mở

`macro.omo()` là bề mặt riêng cho hoạt động bơm/hút của Ngân hàng Nhà nước.

In [11]:
omo_tho = client.macro.omo(start=lui_ngay(HOM_NAY, nam=1))
print(f"Các loại nghiệp vụ: {sorted(omo_tho['kind'].unique())}")
print(f"Đơn vị xuất hiện: {sorted(omo_tho['unit'].dropna().unique())}   ← lại là một CỘT")

bom_hut = client.macro.omo(kind="net_pump", start=lui_ngay(HOM_NAY, thang=6))
print(f"\nnet_pump: {len(bom_hut)} dòng · các công cụ: {sorted(bom_hut['name'].unique())}")

Các loại nghiệp vụ: ['interbank_rate', 'interbank_turnover', 'net_pump', 'policy_rate', 'reverse_repo']
Đơn vị xuất hiện: ['%', 'VND']   ← lại là một CỘT

net_pump: 254 dòng · các công cụ: ['Bơm hút ròng Reverse Repo', 'Bơm hút ròng tín phiếu']


In [12]:
theo_ngay = (
    bom_hut.groupby("date", observed=True)["value"].sum().div(1e12).rename("nghin_ty").reset_index()
)
print(f"Bơm/hút ròng 6 tháng: tổng {theo_ngay['nghin_ty'].sum():+,.1f} nghìn tỷ đồng")

thanh_doi_mau(
    theo_ngay.tail(40).assign(nhan=lambda d: d["date"].dt.strftime("%d/%m")),
    x="nhan",
    y="nghin_ty",
    tieu_de="NHNN bơm/hút ròng qua thị trường mở — 40 phiên gần nhất",
    phu_de="Dương là bơm tiền ra, âm là hút tiền về · cộng cả reverse repo và tín phiếu",
    nhan_y="nghìn tỷ đồng",
    dinh_dang_nhan="{:+.0f}",
)

Bơm/hút ròng 6 tháng: tổng -294.1 nghìn tỷ đồng


In [13]:
luy_ke = theo_ngay.assign(tich_luy=theo_ngay["nghin_ty"].cumsum())

duong(
    luy_ke,
    x="date",
    y="tich_luy",
    tieu_de="Bơm/hút ròng tích luỹ — 6 tháng",
    phu_de="Xu hướng của đường này là trạng thái thanh khoản hệ thống, không phải giá trị một phiên",
    nhan_y="nghìn tỷ đồng, tích luỹ",
    moc_khong=True,
)

## 6 · Cán cân thương mại

In [14]:
xnk = pd.concat(
    [
        client.macro.trade(flow="export", start=lui_ngay(HOM_NAY, nam=3)).assign(nhan="Xuất khẩu"),
        client.macro.trade(flow="import", start=lui_ngay(HOM_NAY, nam=3)).assign(nhan="Nhập khẩu"),
    ]
)
can_can = client.macro.trade(flow="balance", start=lui_ngay(HOM_NAY, nam=3))

print(f"Đơn vị: {sorted(xnk['unit'].unique())}")
print(f"Kỳ gần nhất {can_can['period'].iloc[-1]}: cán cân {can_can['value'].iloc[-1] / 1e9:+,.2f} tỷ USD")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.6, 0.4])
for nhan, mau in [("Xuất khẩu", CHUOI[0]), ("Nhập khẩu", CHUOI[1])]:
    d = xnk[xnk["nhan"] == nhan].sort_values("date")
    fig.add_trace(
        go.Scatter(x=d["date"], y=d["value"] / 1e9, name=nhan, line=dict(width=2, color=mau),
                   hovertemplate="%{y:,.2f} tỷ USD<extra>" + nhan + "</extra>"),
        row=1, col=1,
    )
fig.add_trace(
    go.Bar(x=can_can["date"], y=can_can["value"] / 1e9, name="Cán cân",
           marker=dict(color=[TANG if v >= 0 else GIAM for v in can_can["value"]], line=dict(width=0)),
           hovertemplate="%{y:+,.2f} tỷ USD<extra>cán cân</extra>"),
    row=2, col=1,
)
fig.add_hline(y=0, line_width=1, line_color="#c3c2b7", row=2, col=1)
fig.update_yaxes(title_text="tỷ USD / tháng", row=1, col=1)
fig.update_yaxes(title_text="tỷ USD", row=2, col=1)
fig.update_layout(title_text="Xuất nhập khẩu và cán cân thương mại — 3 năm", height=660, hovermode="x unified")
fig

Đơn vị: ['USD']
Kỳ gần nhất 7-2026: cán cân -3.59 tỷ USD


In [15]:
doi_tac = client.macro.trade(flow="export", by="country", start=lui_ngay(HOM_NAY, thang=2))
ky_cuoi = doi_tac["period"].iloc[-1]
top = (
    doi_tac[doi_tac["period"] == ky_cuoi]
    .nlargest(12, "value")
    .assign(ty_usd=lambda d: (d["value"] / 1e9).round(2))
)

bar_ngang(
    top,
    nhan="partner",
    gia_tri="ty_usd",
    tieu_de=f"12 thị trường xuất khẩu lớn nhất — kỳ {ky_cuoi}",
    phu_de=f"Tổng {len(doi_tac[doi_tac['period'] == ky_cuoi])} đối tác trong kỳ",
    nhan_x="tỷ USD",
    dinh_dang_nhan="{:.2f}",
)

## 7 · Vĩ mô đặt cạnh VNINDEX

Câu hỏi cuối, và là lý do notebook này tồn tại: các biến vĩ mô có liên hệ gì
với thị trường cổ phiếu?

⚠️ **Tần suất khác nhau là cái bẫy.** CPI theo tháng, VNINDEX theo phiên. Ghép
chúng phải **đưa giá về tần suất tháng**, không phải điền ngược CPI vào từng
phiên — điền ngược sẽ tạo ra hàng nghìn quan sát giả và làm mọi hệ số tương
quan phồng lên.

In [16]:
vni = client.eod.index.ohlcv("VNINDEX", start=lui_ngay(HOM_NAY, nam=3)).sort_values("date")

vni_thang = (
    vni.assign(ky=vni["date"].dt.to_period("M"))
    .groupby("ky", observed=True)["close"]
    .last()
    .rename("vnindex")
)
vni_thang_ls = (vni_thang.pct_change() * 100).rename("vnindex_ls")

print(f"VNINDEX: {len(vni):,} phiên → {len(vni_thang)} quan sát tháng")
print("→ Ghép ở tần suất tháng, không phải phiên.")

VNINDEX: 747 phiên → 37 quan sát tháng
→ Ghép ở tần suất tháng, không phải phiên.


In [17]:
bien_thang = {}
for nhan in ["CPI so cùng kỳ", "Tín dụng so cùng kỳ", "Tỷ giá trung tâm", "Lãi suất qua đêm", "Dầu Brent"]:
    d = lay(nhan)
    if d.empty:
        continue
    bien_thang[nhan] = (
        d.assign(ky=d["date"].dt.to_period("M")).groupby("ky", observed=True)["value"].last()
    )

bang = pd.DataFrame(bien_thang).join(vni_thang).join(vni_thang_ls).dropna()
print(f"{len(bang)} tháng có đủ mọi biến")
bang.tail(6).round(2)

35 tháng có đủ mọi biến


,CPI so cùng kỳ,Tín dụng so cùng kỳ,Tỷ giá trung tâm,Lãi suất qua đêm,Dầu Brent,vnindex,vnindex_ls
ky,,,,,,,
2026-02,3.35,19.74,25044.0,4.70,72.48,1880.33,2.80
2026-03,4.65,18.24,25102.0,9.57,118.35,1674.49,-10.95
2026-04,5.46,18.25,25113.0,3.88,114.01,1854.10,10.73
2026-05,5.60,18.23,25139.0,6.97,92.05,1863.49,0.51
2026-06,4.69,17.41,25206.0,12.49,72.92,1860.01,-0.19
2026-07,4.45,17.05,25338.0,5.72,90.12,1735.78,-6.68


In [18]:
tuong_quan = bang.corr(numeric_only=True)["vnindex_ls"].drop(["vnindex", "vnindex_ls"]).round(3)
print("Tương quan MỨC vĩ mô với lợi suất tháng của VNINDEX:\n")
print(tuong_quan.to_string())
print(f"\n(cỡ mẫu {len(bang)} tháng — sai số chuẩn xấp xỉ {1 / np.sqrt(len(bang)):.3f})")

Tương quan MỨC vĩ mô với lợi suất tháng của VNINDEX:

CPI so cùng kỳ        -0.089
Tín dụng so cùng kỳ    0.246
Tỷ giá trung tâm       0.113
Lãi suất qua đêm      -0.244
Dầu Brent             -0.288

(cỡ mẫu 35 tháng — sai số chuẩn xấp xỉ 0.169)


⚠️ **Đọc bảng này rất thận trọng.** Với 36 quan sát, sai số chuẩn của một hệ số
tương quan là khoảng 0,17 — nghĩa là mọi hệ số dưới 0,33 không phân biệt được
với 0. Và tương quan giữa một **mức** với một **lợi suất** dễ bị nhiễu bởi xu
hướng chung của cả hai chuỗi.

Kết luận đúng đắn từ ba năm dữ liệu tháng là: **không kết luận gì cả**. Vĩ mô
tác động lên thị trường qua nhiều kênh với độ trễ khác nhau, và ba năm là 36
điểm dữ liệu.

In [19]:
chuan_hoa = pd.concat(
    [
        pd.DataFrame({"ky": bang.index.to_timestamp(), "gia_tri": bang["vnindex"] / bang["vnindex"].iloc[0] * 100, "chuoi": "VNINDEX"}),
        pd.DataFrame({"ky": bang.index.to_timestamp(), "gia_tri": bang["Tín dụng so cùng kỳ"], "chuoi": "Tín dụng YoY (%)"}),
        pd.DataFrame({"ky": bang.index.to_timestamp(), "gia_tri": bang["CPI so cùng kỳ"], "chuoi": "CPI YoY (%)"}),
    ]
)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.55, 0.45])
fig.add_trace(
    go.Scatter(x=bang.index.to_timestamp(), y=bang["vnindex"], name="VNINDEX",
               line=dict(width=2, color=CHUOI[0])),
    row=1, col=1,
)
for i, cot in enumerate(["Tín dụng so cùng kỳ", "CPI so cùng kỳ"]):
    fig.add_trace(
        go.Scatter(x=bang.index.to_timestamp(), y=bang[cot], name=cot,
                   line=dict(width=2, color=CHUOI[i + 1])),
        row=2, col=1,
    )
fig.update_yaxes(title_text="điểm chỉ số", row=1, col=1)
fig.update_yaxes(title_text="%", row=2, col=1)
fig.update_layout(
    title_text="VNINDEX và hai biến vĩ mô<br>"
    "<sub style='color:#52514e'>Điểm chỉ số và phần trăm là hai đơn vị — hai khung, không phải hai trục y</sub>",
    height=660, hovermode="x unified",
)
fig

## 8 · Dashboard vĩ mô một trang

In [20]:
THU_MUC_RA = GOC / "output"
THU_MUC_RA.mkdir(exist_ok=True)

CSS = """
<style>
.bang-o { display:flex; gap:12px; flex-wrap:wrap; font-family:system-ui,-apple-system,"Segoe UI",sans-serif; margin:8px 0 20px; }
.o { flex:1 1 175px; background:#fcfcfb; border:1px solid rgba(11,11,11,.10); border-radius:10px; padding:14px 16px; }
.nhan { font-size:12px; color:#52514e; }
.gia-tri { font-size:24px; font-weight:600; margin:4px 0 2px; color:#0b0b0b; }
.phu { font-size:12px; color:#898781; }
</style>
"""


def o_vi_mo(nhan: str, gia_tri: float, don_vi: str, ky: str, truoc: float | None) -> str:
    if truoc is None or pd.isna(truoc):
        mo_ta = ky
    else:
        delta = gia_tri - truoc
        mui = "▲" if delta > 0 else ("▼" if delta < 0 else "—")
        mau = TANG if delta > 0 else (GIAM if delta < 0 else "#898781")
        mo_ta = f"{ky} · <span style='color:{mau}'>{mui} {abs(delta):,.2f}</span> so kỳ trước"
    so = f"{gia_tri:,.2f}" if abs(gia_tri) < 1000 else f"{gia_tri:,.0f}"
    return f'<div class="o"><div class="nhan">{nhan}</div><div class="gia-tri">{so} <span style="font-size:13px;color:#898781">{don_vi}</span></div><div class="phu">{mo_ta}</div></div>'


cac_o = []
for nhan in ["CPI so cùng kỳ", "GDP tăng trưởng", "Tín dụng so cùng kỳ", "PMI", "Tỷ giá trung tâm", "Lãi suất qua đêm"]:
    d = lay(nhan)
    if d.empty:
        continue
    cac_o.append(
        o_vi_mo(
            nhan,
            float(d["value"].iloc[-1]),
            str(d["unit"].iloc[-1]),
            str(d["period"].iloc[-1]),
            float(d["value"].iloc[-2]) if len(d) > 1 else None,
        )
    )

from IPython.display import HTML

HTML(CSS + f'<div class="bang-o">{"".join(cac_o)}</div>')

⚠️ Mỗi ô mang **đơn vị của chính nó**, đọc từ cột `unit` của dòng đó. Đây
không phải chi tiết trình bày — nó là hệ quả trực tiếp của việc `macro` để
`unit` ở mức dòng, và là cách duy nhất để một dashboard trộn `%`, `VND` và
`Lần` mà không nói dối người đọc.

In [21]:
def xuat_dashboard(duong_dan: Path) -> Path:
    khoi = [
        fig.to_html(full_html=False, include_plotlyjs=True),
    ]
    html = f"""<!doctype html>
<html lang="vi"><head><meta charset="utf-8">
<title>Dashboard vĩ mô — {HOM_NAY:%d/%m/%Y}</title>
{CSS}
<style>
  body {{ background:#f9f9f7; color:#0b0b0b; margin:0; padding:28px 32px 48px;
         font-family:system-ui,-apple-system,"Segoe UI",sans-serif; }}
  h1 {{ font-size:22px; margin:0 0 2px; }}
  .moc {{ color:#898781; font-size:13px; margin-bottom:18px; }}
  .bieu-do {{ background:#fcfcfb; border:1px solid rgba(11,11,11,.10);
              border-radius:10px; padding:8px; margin-bottom:16px; }}
  footer {{ color:#898781; font-size:12px; margin-top:24px;
            border-top:1px solid #e1e0d9; padding-top:12px; }}
</style></head>
<body>
  <h1>Dashboard vĩ mô Việt Nam</h1>
  <div class="moc">Cập nhật {HOM_NAY:%d/%m/%Y} · nguồn Tổng cục Thống kê và Ngân hàng Nhà nước qua FinLens ·
       mỗi chỉ tiêu mang đơn vị riêng của nó</div>
  <div class="bang-o">{"".join(cac_o)}</div>
  {"".join(f'<div class="bieu-do">{k}</div>' for k in khoi)}
  <footer>Sinh tự động từ notebook <code>44_vi_mo_va_thi_truong.ipynb</code></footer>
</body></html>"""
    duong_dan.write_text(html, encoding="utf-8")
    return duong_dan


tep = xuat_dashboard(THU_MUC_RA / f"dashboard_vi_mo_{HOM_NAY:%Y%m%d}.html")
print(f"Đã ghi: {tep.name} ({tep.stat().st_size / 1024:,.0f} KB)")

Đã ghi: dashboard_vi_mo_20260811.html (4,751 KB)

## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Tra mã chỉ tiêu | `macro.indicators(topic=..., freq=...)` |
| Lấy số liệu | `macro.series([ma1, ma2, ...])` |
| Bơm/hút của NHNN | `macro.omo(kind="net_pump")` |
| Xuất nhập khẩu | `macro.trade(flow="balance")` · `by="country"` |

**Bốn điều đáng nhớ:**

1. ⚠️ **`unit` là một CỘT**, không phải thuộc tính của bảng. Mọi phép tính phải
   bắt đầu bằng `groupby("code")` hoặc một bộ lọc `code ==`.
2. `topic="cpi"` **không ném lỗi** — nó trả về bảng rỗng đúng schema. Tra mã
   bằng cách lọc danh mục, và **khẳng định tìm thấy**.
3. `date` là **cuối kỳ quan sát**, không phải một phiên giao dịch. `period` mới
   nói kỳ nào.
4. Ghép vĩ mô với giá phải **hạ giá xuống tần suất của vĩ mô**, không điền
   ngược vĩ mô lên từng phiên.

---

## Hết 17 notebook

Bạn đã đi qua toàn bộ bề mặt của `finlens 1.3.0`: giá EOD và intraday, dòng
tiền nhà đầu tư, báo cáo tài chính và 181 chỉ tiêu, 135 hàm TA-Lib, phái sinh,
chứng quyền, và 3.348 chuỗi vĩ mô.

**Thứ đáng mang theo nhất không phải danh sách hàm** — nó nằm trong `help()`.
Thứ đáng mang theo là bốn thói quen:

| Thói quen | Notebook chứng minh |
|---|---|
| Đọc đơn vị từ dữ liệu, đừng nhớ | `00`, `43` — bẫy 1000 lần |
| Chỉ báo trên frame nhiều mã phải tách nhóm | `31` — sai hàng trăm dòng, không cảnh báo |
| Trừ tần suất nền trước khi kết luận | `32`, `33` — tín hiệu kêu 30% số phiên không phải tín hiệu |
| Đo giá của mỗi giả định trong backtest | `34` — ba lỗi biến −5,7% thành +3,0% |

Bốn thói quen đó không phụ thuộc vào thư viện nào. Chúng là phần khó của công
việc; phần còn lại chỉ là gọi hàm.